# EnzyExtract

This tutorial describes EnzyExtract, a pipeline to extract $k_{cat}$ and $K_M$ values using LLMs.

It accompanies the manuscript, *Finding the Dark Matter: Large Language Model-based Enzyme Kinetic Data Extractor and Its Validation*. A preprint is available on [chemrxiv](https://chemrxiv.org/engage/chemrxiv/article-details/6801df4850018ac7c5f340a1).

## Step ii: Download papers

Place your papers in /content/papers. Alternatively, use the example paper.

The example paper is *Arsinothricin, an arsenic-containing
 non-proteinogenic amino acid analog of glutamate,
 is a broad-spectrum antibiotic*, by Nadar et al., which is available under [CC BY 4.0](http://creativecommons.org/licenses/by/4.0/).


 Papers should be given a unique filename, and we recommend using the PubMed ID.


In [ ]:
!mkdir /content/papers
!wget -O /content/papers/30993215.pdf https://www.nature.com/articles/s42003-019-0365-y.pdf

## Step i: Installation





In [ ]:
!pip install git+https://github.com/conjuncts/gmft_pymupdf -q

In [ ]:
!git clone https://github.com/ChemBioHTP/EnzyExtract
%cd EnzyExtract
!git pull origin main
!pip install -e .
%cd /content/

In [ ]:
!pwd

In [ ]:
!pip show enzyextract

Note: you may need to follow [these](
https://stackoverflow.com/questions/57838013/modulenotfounderror-after-successful-pip-install-in-google-colaboratory) special steps for the editable install.


In [ ]:
import site
site.main()
import importlib

In [ ]:
# reload modules
import sys
def refresh_modules():
    _modules = sys.modules.copy()
    for module in _modules.values():
        if 'enzyextract' in str(module):
            importlib.reload(module)
refresh_modules()

## Step 0: Preprocessing

Code can be found in `experiments/example/pipeline/ex_step0_run_preprocessing.py`.

Click the "eye" in the left panel to see the hidden `.enzy` folder. [See here](https://stackoverflow.com/questions/67698933/how-to-show-hidden-files-colab).

In [ ]:
import os
from enzyextract.pre.reocr.m_mu_reocr import script_scan_mM
from enzyextract.pre.scans.scan_to_parquet import scan_papers
from enzyextract.pre.table.scan_tables import process_pdfs

if __name__ == '__main__':

    pdf_root = '/content/papers' # PDFs to process
    enzy_root = '/content/.enzy' # where intermediate data for these PDFs is stored

    print("Starting mM...")
    script_scan_mM(
        pdf_root=pdf_root,
        write_dir=f'{enzy_root}/pre/mM',
        model_path='EnzyExtract/data/models/resnet18-remicro-iter3.pth',
    )

    print("Starting tables...")
    process_pdfs(
        pdf_root=pdf_root,
        write_dir=f"{enzy_root}/pre/tables",
        micros_path=f"{enzy_root}/pre/mM/mM.parquet",
        # _check_nonzero_tables=False,
    )

    print(f"Compressing PDFs to {enzy_root}/scans/pdf/pdf.parquet")
    df = scan_papers(
        pdfs_folder=pdf_root,
        recursive=False,
    )
    os.makedirs(f'{enzy_root}/scans/pdf', exist_ok=True)
    df.write_parquet(f'{enzy_root}/scans/pdf/pdf.parquet')

In [ ]:
!ls /content/papers

In [ ]:
from IPython.display import display, Markdown

with open("/content/.enzy/pre/tables/markdown/30993215_0.md") as f:
  display(Markdown(f.read()))

## Step 1: Submission

You will need to set your `OPENAI_API_KEY` in Google Colab's Secrets panel. You may need to [sign up](https://platform.openai.com/api-keys) with OpenAI. You can also call `process_env('.env')`.

By default, EnzyExtract uses the Batch API.

The batch API is the best method for large volumes of papers, since OpenAI and other vendors offer a 50% discount and higher volumes can be processed than synchronously.

However, the Batch API does not work well with Google Colab. The colab session may time out before the batch completes, in which case the `.enzy` metadata and correspondences between custom_ids and PMIDs (`.enzy/corresp`) will be lost.

When submitting the file, you will have a couple of options.
- `l` (**local**) **is recommended**. This will save a local copy. Then, see the below instructions to run synchronously.
- `y`(**yes**) will use the Batch API. This is recommended for large scale processing, but not for Google Colab.



In [ ]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [ ]:
from enzyextract.pipeline.step1_run_tableboth import process_env, step1_main
from enzyextract.utils import prompt_collections

if __name__ == '__main__':

    llm_provider = 'openai'
    model_name = 'gpt-4o-2024-08-06'
    suggested_prompt = prompt_collections.table_oneshot_v3
    structured = False

    namespace = 'my-namespace-here' # no colons: needs to be a valid file name
    pdf_root = '/content/papers'
    enzy_root = '/content/.enzy'
    step1_main(
        namespace=namespace,
        pdf_root=pdf_root,
        micro_path=f'{enzy_root}/pre/mM/mM.parquet',
        tables_from=f'{enzy_root}/pre/tables/markdown',

        dest_folder=f'{enzy_root}/batches',
        corresp_folder=f'{enzy_root}/corresp',
        log_location=f'{enzy_root}/llm_log.tsv',
        model_name=model_name,
        llm_provider=llm_provider,
        prompt=suggested_prompt,
        structured=structured,

        # _check_nonzero_tables=False,
        _check_nonzero_reocr=False,
    )

Check your batches like so.

In [ ]:
from enzyextract.pipeline.llm_log import read_log
read_log('/content/.enzy/llm_log.tsv')

## Step 2: Download

If you are in Google Colab and are processing synchronously, see below.

In [ ]:
from enzyextract.submit.openai_synch import process_batch_synchronously

process_batch_synchronously(
    batch_fpath='/content/.enzy/batches/my-namespace-here_v1.jsonl',
    enzy_root='/content/.enzy'
)

In [ ]:
from enzyextract.pipeline.llm_log import read_log
read_log('/content/.enzy/llm_log.tsv')

### Option 2b (Batch API)

If you used the Batch API, see below.

In [ ]:
from enzyextract.pipeline.step2_download import process_env, download

if __name__ == "__main__":
    process_env('.env')
    download(
        log_location="/content/.enzy/llm_log.tsv",
        dest_folder="/content/.enzy/completions",
        err_folder="/content/.enzy/errors",
    )

In [ ]:
!zip -r enzy.zip /content/.enzy

## Step 3: Convert to DataFrame

We use:
- polars for faster performance
- parquet for smaller file sizes, null safety, and nested column types

In [ ]:
import os

from enzyextract.pipeline.step3_llm_to_df import namespace_to_parquet

if __name__ == '__main__':
    enzy_root = '/content/.enzy'
    df = namespace_to_parquet(
        namespace='my-namespace-here',
        log_location=f'{enzy_root}/llm_log.tsv',
        write_dir=f'{enzy_root}/post/valid'
    )

In [ ]:
df